In [ ]:
%load_ext autoreload
%autoreload 2

import os
import glob
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from datetime import datetime
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import seaborn as sns
from matplotlib.colors import ListedColormap


import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.config.config import DATA_RAW
from src.loaders.csv_loader import CSVLoader
from src.dataProcessing.data_quality import DataQuality
from src.dataProcessing.data_features import DataFeatures
from src.dataProcessing.data_quality_features import DataQualityFeatures
from src.visualization.TimeBoxplot import SalesTimeBoxplot
from src.visualization.TimeViolin import SalesTimeViolin
from src.visualization.TimeOutlier import SalesTimeOutlier

from src.models.IsolationForest import IsolationForestAnalyzer


csvLoader = CSVLoader()

In [ ]:
df = csvLoader.create_dataframe(DATA_RAW,"synthetic_beverage_sales_data.csv")

In [ ]:
print (df.head())

In [ ]:
# B2C com desconto indevido
invalid_b2c_discount = df[
    (df["Customer_Type"] == "B2C") &
    (df["Discount"] > 0)
]

print(invalid_b2c_discount.head())

In [ ]:
dataQuality = DataQuality(df)

In [ ]:
report = dataQuality.run_full_analysis(tolerance=0.1)
print(report)

In [ ]:
#report = dataQuality.run_full_analysis(tolerance=0.1)
#print(report)

sample = dataQuality.get_total_price_inconsistencies_sample(tolerance=0.1, n=20)
print(sample.to_string(index=False))


In [ ]:
print(dataQuality.generate_executive_summary())

In [ ]:
#DataFeatures
""" df_features = DataFeatures(
    df,
    date_col="Order_Date",
    group_cols=["Product", "Region"],
    windows=(7, 14, 30),
    min_periods=1,
    fill_missing_days=True
)

print(df_features.head())
print(df_features.columns.tolist()) """

In [ ]:
feature_builder = DataQualityFeatures(
    df=df,
    date_col="Order_Date",
    group_cols=["Product", "Region"],
    fill_missing_days=True
)

# Gera as features
df_features = feature_builder.build_all_features()

# Vamos supor que você queira verificar se ainda existe NaN
print(df_features[
    [
        "quantity_sum_std_7d",
        "total_price_sum_std_7d",
        "quantity_vs_mean_7d",
        "total_price_vs_mean_30d",
        "quantity_pct_vs_mean_7d"
    ]
].isna().sum())

In [ ]:
report = feature_builder.run_full_analysis()
print(report)
print(feature_builder.generate_executive_summary())


In [ ]:
print (df_features.dtypes) 

In [ ]:
df_final = feature_builder.fix_missing_feature_values()

In [ ]:
report = feature_builder.run_full_analysis()
print(report)
print(feature_builder.generate_executive_summary())

In [ ]:
boxplot = SalesTimeBoxplot(df)

boxplot.plot_boxplot_by_period(
    value_col="Quantity",
    period="month",
    filters={
        "Category": ["Soft Drinks", "Water", "Juices"]
    },
    start_date="2022-12-01",
    end_date="2022-12-31"
)

In [ ]:
violin_analyzer = SalesTimeViolin(df)

violin_analyzer.plot_violin_by_period(
    value_col="Quantity",
    period="month",
    filters={
        "Category": ["Soft Drinks", "Water", "Juices"]
    },
    start_date="2022-12-01",
    end_date="2022-12-31",
    pastel_color="#D9C2E9",
    use_log_scale=True
)

In [ ]:
violin = SalesTimeViolin(df)

violin.plot_violin_by_period(
    value_col="Quantity",
    period="month",
    filters={
        "Category": ["Soft Drinks", "Water", "Juices"],
        "Region": ["Baden-Württemberg"]
    },
    start_date="2023-12-01",
    end_date="2024-12-31",
    use_log_scale=False
)
''' 
violin_analyzer.violin_analyzer.plot_multiple_violins(
    columns=["Total_Price", "Quantity", "Discount", "Unit_Price"],
    period="year",
    filters={
        "Category": ["Soft Drinks", "Water", "Juices"],
        "Region": ["Baden-Württemberg"]
    },
    start_date="2023-12-01",
    end_date="2023-12-31"
)'''

In [ ]:
outlier_plot = SalesTimeOutlier(df)

resultado = outlier_plot.plot_outliers_by_period(
    value_col="Quantity",
    filters={
         "Category": ["Soft Drinks", "Water", "Juices"],
        "Region": ["Baden-Württemberg"]
    },
    start_date="2022-12-01",
    end_date="2022-12-31",
    agg="sum",
    method="iqr",
    moving_window=None
)

resultado[resultado["is_outlier"]]

In [ ]:
resultado = outlier_plot.plot_outliers_by_period(
    value_col="Quantity",
    filters={
         "Category": ["Soft Drinks", "Water", "Juices"],
        "Region": ["Baden-Württemberg"]
    },
    start_date="2022-10-01",
    end_date="2022-12-31",
    agg="sum",
    method="iqr",
    moving_window=7
)

resultado[resultado["is_outlier"]]

In [ ]:
resultado = outlier_plot.plot_outliers_by_period(
    value_col="Quantity",
    filters={
        "Category": ["Juices"]
    },
    start_date="2022-12-01",
    end_date="2022-12-31",
    agg="sum",
    method="zscore",
    z_threshold=3.0
)

resultado[resultado["is_outlier"]]

In [ ]:
if_analyzer = IsolationForestAnalyzer(
    random_state=42,
    cv=3,
    n_jobs=-1,
    verbose=1
)

if_analyzer.fit(df_final)

print(if_analyzer.get_best_params())

df_if = if_analyzer.predict(df_final)

print(
    df_if[
        [
            "Order_Date",
            "Product",
            "Region",
            "if_qty_signal",
            "if_sales_signal",
            "if_discount_signal",
            "if_ticket_signal",
            "anomaly_flag",
            "anomaly_score"
        ]
    ].head()
)

summary_if = if_analyzer.anomaly_summary(df_if)
print(summary_if.head(20))

In [ ]:
df_if.head()

In [ ]:
top_water = if_analyzer.top_n_anomalies(
    df_pred=df_if,
    n=10,
    product_filter="Evian"
)

print(top_water)

In [ ]:
if_analyzer.plot_anomalies_over_time(
    df_pred=df_if,
    date_col="Order_Date",
    value_col="total_price_sum",
    title="Anomalies - Evian",
    product_filter="Evian"
)

In [ ]:
if_analyzer.plot_anomalies_subplots(
    df_pred=df_if,
    group_by="Region",
    date_col="Order_Date",
    value_col="avg_ticket",
    max_groups=4
)

In [ ]:
if_analyzer.plot_anomalies_subplots(
    df_pred=df_if,
    group_by="Product",
    date_col="Order_Date",
    value_col="total_price_sum",
    region_filter="South",
    max_groups=5
)

In [ ]:
if_analyzer.save_model()

In [ ]:
if_analyzer.save_best_params_json()

####################

In [ ]:
print(if_analyzer.best_params_)


In [ ]:
print(if_analyzer.best_estimator_)

In [ ]:
import os
import joblib
from datetime import datetime


def save_isolation_forest_analyzer(model, folder_path="models/isolation_forest", file_name=None):
    """
    Salva em disco uma instância já treinada da classe IsolationForestAnalyzer.

    Parâmetros
    ----------
    model : object
        Instância treinada da classe IsolationForestAnalyzer.
        É esperado que já tenha sido executado model.fit(df).

    folder_path : str, default="models/isolation_forest"
        Pasta onde o modelo será salvo.

    file_name : str ou None, default=None
        Nome do arquivo .joblib.
        Se None, será gerado automaticamente com timestamp.

    Retorno
    -------
    str
        Caminho completo do arquivo salvo.
    """
    if not hasattr(model, "fitted_") or not model.fitted_:
        raise ValueError(
            "O modelo ainda não foi treinado. Execute fit(df) antes de salvar."
        )

    os.makedirs(folder_path, exist_ok=True)

    if file_name is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        file_name = f"isolation_forest_analyzer_{timestamp}.joblib"

    full_path = os.path.join(folder_path, file_name)

    joblib.dump(model, full_path) #O compress =2 é para tentar diminuir o modelo para menos 100Mb

    print(f"Modelo salvo com sucesso em: {full_path}")
    return full_path


def load_isolation_forest_analyzer(model_path):
    """
    Carrega de disco uma instância salva da classe IsolationForestAnalyzer.

    Parâmetros
    ----------
    model_path : str
        Caminho completo do arquivo .joblib salvo anteriormente.

    Retorno
    -------
    object
        Instância carregada da classe IsolationForestAnalyzer.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Arquivo não encontrado: {model_path}")

    model = joblib.load(model_path)
    print(f"Modelo carregado com sucesso de: {model_path}")
    return model

In [ ]:
model_path = save_isolation_forest_analyzer(
    if_analyzer,
    folder_path="models/isolation_forest"
)

In [ ]:
import json


def save_best_params_json(model, folder_path="models/isolation_forest", file_name="best_params.json"):
    """
    Salva os melhores hiperparâmetros encontrados no GridSearch em um arquivo JSON.
    """
    if not hasattr(model, "best_params_") or model.best_params_ is None:
        raise ValueError("O modelo não possui best_params_. Execute fit(df) antes.")

    os.makedirs(folder_path, exist_ok=True)

    full_path = os.path.join(folder_path, file_name)

    with open(full_path, "w", encoding="utf-8") as f:
        json.dump(model.best_params_, f, indent=4, ensure_ascii=False)

    print(f"Best params salvos em: {full_path}")
    return full_path

In [ ]:
save_best_params_json(if_analyzer)